In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.nn.functional as F
import tqdm
import matplotlib.pyplot as plt
import torchvision

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Dataset

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

train_dataset = datasets.CIFAR10(root='data', train=True, transform=transform, download=True)
train_loader = DataLoader(dataset=train_dataset, batch_size=128, shuffle=True)

# VQ embedding

In [ ]:
class VQEmbedding(nn.Module):
    def __init__(self, embedding_dim, codebook_length, beta=0.25):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.num_embeddings = codebook_length
        self.beta = beta
        self.embedding = nn.Embedding(codebook_length, embedding_dim)
        self.embedding.weight.data.uniform_(-1/codebook_length, 1/codebook_length)

    def forward(self, z):
        b, c, h, w = z.shape
        z_channel_last = z.permute(0, 2, 3, 1)
        z_flattened = z_channel_last.reshape(b * h * w, self.embedding_dim)
        distances = torch.cdist(z_flattened, self.embedding.weight, p=2)
        encoding_indices = torch.argmin(distances, dim=-1)
        z_q = self.embedding(encoding_indices)
        z_q = z_q.reshape(b, h, w, self.embedding_dim).permute(0, 3, 1, 2)
        vq_loss = F.mse_loss(z_q, z.detach()) + self.beta * F.mse_loss(z_q.detach(), z)
        z_q = z + (z_q - z).detach()
        return z_q, vq_loss, encoding_indices

# VQ-VAE

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, channels):
        super(ResBlock, self).__init__()
        self.block = nn.Sequential(
            nn.ReLU(inplace=False),
            nn.Conv2d(channels, channels, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=False),
            nn.Conv2d(channels, channels, kernel_size=3, stride=1, padding=1)
        )
    def forward(self, x):
        return x + self.block(x)

class VQVAE(nn.Module):
    def __init__(self, channels, latent_dim, codebook_length):
        super(VQVAE, self).__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(channels, 128, kernel_size=4, stride=2, padding=1),  
            nn.ReLU(inplace=False),
            nn.Conv2d(128, latent_dim, kernel_size=4, stride=2, padding=1),  
            nn.ReLU(inplace=False),
            ResBlock(latent_dim),
            ResBlock(latent_dim)
        )
        self.vq_layer = VQEmbedding(latent_dim, codebook_length)
        self.decoder = nn.Sequential(
            ResBlock(latent_dim),
            ResBlock(latent_dim),
            nn.ConvTranspose2d(latent_dim, 128, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=False),
            nn.ConvTranspose2d(128, channels, kernel_size=4, stride=2, padding=1),
            nn.Tanh()
        )

    def forward(self, x):
        z_e = self.encoder(x)
        z_q, vq_loss, encoding_indices = self.vq_layer(z_e)
        x_recon = self.decoder(z_q)
        return x_recon, vq_loss

# Entrenamiento

In [ ]:
def train(model, train_loader, optimizer, num_epochs=50):
    model.train()
    pbar = tqdm.trange(num_epochs)
    for epoch in pbar:
        train_loss = 0
        for x, _ in train_loader:
            x = x.to(device)
            optimizer.zero_grad()
            x_recon, vq_loss = model(x)
            recon_loss = F.mse_loss(x_recon, x)
            loss = recon_loss + vq_loss
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        avg_loss = train_loss / len(train_loader)
        pbar.set_postfix({'Loss': avg_loss})

In [ ]:
model = VQVAE(channels=3, latent_dim=128, codebook_length=512).to(device)
optimizer = optim.Adam(model.parameters(), lr=2e-4)
train(model, train_loader, optimizer)

# Reconstrucción

In [ ]:
def show_image(batch_of_tensors):
    images = batch_of_tensors[:4]
    images = (images * 0.5) + 0.5  
    grid_img = torchvision.utils.make_grid(images, nrow=2)
    plt.figure(figsize=(5, 5))
    plt.imshow(grid_img.permute(1, 2, 0))
    plt.axis('off')
    plt.show()

x, _ = next(iter(train_loader))
x = x.to(device)
with torch.no_grad():
    recon_batch, _ = model(x)
    show_image(x.cpu())
    show_image(recon_batch.cpu())